# Timing &amp; scaling — serial vs batched (CPU) vs batched (GPU) forward solve

**Reviewer answer R1-C8 / R2-m2.** Times the atomic unit of the EFC method — one
`runTime`-second forward ODE solve of the cardio model — executed three ways across a
cohort-size sweep `N`, to report **s/subject** and how wall-clock **scales** with the
number of subjects. A full staged calibration is just `K` such solves (its absolute cost
is the 80.3 s/calibration Euler figure from `solver_compare`); this notebook isolates the
*scaling lever*: batching a cohort onto one `jax.vmap`.

Three legs: **serial** (`batchedBaselineSI` at N=1 in a Python loop), **batch-CPU** and
**batch-GPU** (`batchedBaselineSI` at N, one vmap). GPU selection is a pre-`import jax`
env switch that cannot be reflipped mid-kernel, so the GPU leg is a **second kernel run**
(`device.useGpu=True`, `timing.modes=["batch"]`) writing `timing_gpu.h5`; Phase 2 merges
both files.

<details>
<summary>Two-phase, single-config-surface driver (like the other <code>run_convergence</code> notebooks)</summary>

Phase 1 runs the sweep for the current device and persists per-cell wall-clock via
`schema_pop.write_timings` (the grid labels ride in the `meta["cells"]` context). Phase 2
loads `timing_cpu.h5` (+ `timing_gpu.h5` if present), builds the s/subject + speedup table,
and emits the paper table/figure. Every knob lives in the single `runConfig` dict.
</details>

In [ ]:
# region -> runConfig — the single run-configuration surface (device/precision applied before JAX)
####################################################################################################
# runConfig — the ONE place run configuration lives (project rule: see repo-root CLAUDE.md).
# Defined first so the device/precision block can be applied before JAX initialises below.
####################################################################################################
runConfig = {
    # --- file references ---------------------------------------------------
    "model":    "cvModel.json",     # config/models/ cardiopulmonary model
    "scenario": "sepsis_linear.json",   # config/scenarios/ tiny fixture (10 s solve) — the unit of work
    "mode":     "calibration",         # one forward solve, no calibration / no convergence block needed

    # --- pipeline phases (run + analyse separable) -------------------------
    "run":  False,    # Phase 1 — execute the timing sweep for THIS device + save timing_<device>.h5
    "plot": True,    # Phase 2 — load timing_{cpu,gpu}.h5 + analyse + emit paper table/figure

    # --- device / precision (applied in the Imports cell, before `import jax`) ---------
    # GPU leg = rerun the whole notebook in a FRESH kernel with useGpu=True (JAX platform
    # can't be reflipped mid-process). It writes timing_gpu.h5; Phase 2 merges it with the CPU file.
    "device": {
        "useGpu":    False,       # committed False (GPU HARD RULE); flip True only for the GPU pass
        "precision": "float64",   # run precision: "float64" (reference) or "float32"
    },

    # --- integration stack + solver ---------------------------------------
    "stack":  "SI",               # step-independent stack (batchedBaselineSI vmaps its euler/rk4 solver)
    "solver": {"type": "euler"},  # euler | rk4 (both vmappable); adaptive solvers don't vmap
    "chunkSize": 256,             # batch leg: samples per vmap (VRAM bound); <=0 = all N in one vmap

    # --- timing sweep (the knobs consumed in Phase 1) ----------------------
    "timing": {
        "nrModelsSweep": [1, 8, 64, 256, 1024],   # #subjects axis; serial's largest N dominates wall time
        "modes":         ["batch"],     # CPU kernel: both; GPU kernel -> set ["batch"]
        "repeats":       3,        # timed passes per cell; report the median (drops scheduler jitter)
        "warmup":        True,     # one untimed pass first, so JIT-compile time is excluded
    },

    # --- output ------------------------------------------------------------
    # `name` satisfies buildSimulationParams (per-run sim-artifact path, unused here since
    # postProcessing is None); the timing files are written as timing_{cpu,gpu}.h5 under `path`.
    "output": {"save": True, "path": "notebookData/test", "name": "timing2.h5"},

    # --- paper artifacts (R1-C8 / R2-m2 timing + scaling) ------------------
    "paper": {
        "emit":         True,
        "generatedDir": "EFC_Paper/revision/generated",
        "imagesDir":    "EFC_Paper/revision/Images",
        "tableName":    "timingComparison2.tex",
        "figName":      "timingComparison2.png",
    },

    "postProcessing": None,
    "plots":          [],
    "printStatus":    True,

    # --- integration numerics (override scenario shared.integration; single config surface) ---
    "runTime": 10,       # simulated seconds per solve (the atomic unit)
    "dt":      0.0005,   # integrator step
    "dtDense": 0.01,     # dense-output sampling step (unused by the final-only timing solve)
}
# endregion


## Imports

In [ ]:
# region -> imports + device/precision (must precede `import jax`)
# ---- repo-root bootstrap: run from any cwd (make `library` importable + resolve
# ---- the relative notebookData/ + config/ paths). Walks up to the dir containing library/. ----
import os, sys
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "library")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)

# ---- device / precision (from runConfig, MUST run before JAX initialises) -------
useGpu    = runConfig["device"]["useGpu"]
precision = runConfig["device"]["precision"]

if useGpu:
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)
    os.environ["JAX_PLATFORMS"] = "cuda"
    os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
    os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]   = "platform"
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
    os.environ["JAX_PLATFORMS"] = "cpu"

import jax
jax.config.update("jax_enable_x64", precision == "float64")

import library.run.runner as runner
import library.run.runnerBatchSI as rb            # batchedBaselineSI: batched forward solve (SI stack)
import library.utils as utils
from library.hdf5 import schema_pop               # write_timings / read_timings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, time

np.set_printoptions(suppress=True)
print("devices:", jax.devices(), "| x64:", jax.config.jax_enable_x64)
# endregion


## Assemble `simulationParams` + build the baseline model once

In [ ]:
# region -> assemble simulationParams + build the SI baseline model once (shared by every cell)
scenario = utils.loadScenario(runConfig["scenario"])
sp = runner.buildSimulationParams(runConfig, scenario)

# Scalar (sample-independent) build: initialiseModel -> configureStates(baseline) -> prepareModel.
# Reused by every (mode, N) cell so nothing rebuilds/recompiles the model per measurement.
prepared = rb.prepareBaseline(sp)
_, _, _, stateNames = prepared

dev = "gpu" if useGpu else "cpu"
outPath = os.path.join(runConfig["output"]["path"], f"timing_{dev}.h5")
print(f"device={dev} | nState={len(stateNames)} | runTime={sp['runTime']}s dt={sp['dt']} "
      f"solver={sp['solver']['type']} | -> {outPath}")
# endregion


# Phase 1 — Run the timing sweep (this device)

For each `N` in `timing.nrModelsSweep` and each `mode` in `timing.modes`, time the median of
`repeats` warm passes. **serial** = `N` sequential `batchedBaselineSI(sp, 1)` calls (one subject
at a time, same SI solver core); **batch** = one `batchedBaselineSI(sp, N, chunkSize)` vmap.
Per-cell total wall + the grid labels are persisted to `timing_<device>.h5`.

<details>
<summary>Cost note</summary>

`serial` at the largest `N` runs `N` solves per pass and dominates wall time (that is exactly
the cost the batch legs improve on). Trim `nrModelsSweep` or drop `repeats` to 1 for a quick pass.
</details>

In [ ]:
# region -> run the (mode x N) timing sweep + persist per-cell wall via schema_pop.write_timings
if runConfig["run"]:
    tcfg     = runConfig["timing"]
    Ns       = list(tcfg.get("nrModelsSweep", [1, 8, 64, 256]))
    modes    = list(tcfg.get("modes", ["serial", "batch"]))
    repeats  = int(tcfg.get("repeats", 3))
    warmup   = bool(tcfg.get("warmup", True))
    chunk    = int(runConfig.get("chunkSize", -1))

    if runConfig["output"]["save"]:
        os.makedirs(runConfig["output"]["path"], exist_ok=True)

    def once(mode, N):
        """One full pass of a cell (blocks internally): serial loops N single-lane solves,
        batch is a single N-wide vmap."""
        if mode == "serial":
            for _ in range(N):
                rb.batchedBaselineSI(sp, 1, prepared=prepared)
        else:
            rb.batchedBaselineSI(sp, N, chunkSize=chunk, prepared=prepared)

    cells, walls = [], []
    t0 = time.time()
    for mode in modes:
        for N in Ns:
            if warmup:                                   # compile the shapes this cell uses (untimed)
                (rb.batchedBaselineSI(sp, 1, prepared=prepared) if mode == "serial"
                 else rb.batchedBaselineSI(sp, N, chunkSize=chunk, prepared=prepared))
            ts = []
            for _ in range(repeats):
                t = time.perf_counter(); once(mode, N); ts.append(time.perf_counter() - t)
            total = float(np.median(ts))
            cells.append({"mode": mode, "N": int(N), "total_wall": total,
                          "s_per_subject": total / N, "repeats": repeats})
            walls.append(total)
            print(f"[{dev}] {mode:6s} N={N:5d} -> {total:8.3f}s total  "
                  f"{1e3 * total / N:8.2f} ms/subject  ({time.time() - t0:.0f}s elapsed)")

    if runConfig["output"]["save"]:
        schema_pop.write_timings(outPath, walls, meta={
            "device":   dev,
            "precision": precision,
            "solver":   sp["solver"]["type"],
            "dt":       sp["dt"],
            "runTime":  sp["runTime"],
            "stack":    runConfig["stack"],
            "chunkSize": chunk,
            "cells":    cells,
            "total_wall": time.time() - t0,
        })
        print(f"\ntimings -> {outPath} ({len(cells)} cells)")
# endregion


# Phase 2 — Load &amp; analyse (merge CPU + GPU files)

Loads `timing_cpu.h5` and, if present, `timing_gpu.h5`, and builds the per-cell table:
device, mode, N, s/subject, total wall, and **speedup vs serial** (CPU serial total at the same N
÷ this cell's total). Independent of Phase 1 having run in this kernel.

In [ ]:
# region -> load timing_{cpu,gpu}.h5 -> tidy dataframe with s/subject + speedup-vs-serial
if runConfig["plot"]:
    rows = []
    for d in ("cpu", "gpu"):
        p = os.path.join(runConfig["output"]["path"], f"timing_{d}.h5")
        if not os.path.exists(p):
            continue
        _, meta = schema_pop.read_timings(p)
        for c in meta.get("cells", []):
            rows.append({"device": meta.get("device", d), "mode": c["mode"], "N": int(c["N"]),
                         "total_wall_s": float(c["total_wall"]),
                         "s_per_subject": float(c["s_per_subject"])})
    df = pd.DataFrame(rows).sort_values(["device", "mode", "N"]).reset_index(drop=True)

    # speedup vs the CPU serial baseline at the same N (the sequential method the batch legs replace).
    # NB bracket access: df["mode"] — attribute df.mode collides with pandas' DataFrame.mode() method.
    serialByN = (df[(df["device"] == "cpu") & (df["mode"] == "serial")]
                 .set_index("N")["total_wall_s"].to_dict())
    df["speedup_vs_serial"] = df.apply(
        lambda r: serialByN.get(r["N"], np.nan) / r["total_wall_s"], axis=1)

    print(f"loaded {len(df)} cells from {sorted(df['device'].unique())}")
    display(df)
# endregion


## Timing table (LaTeX) — R1-C8 / R2-m2

In [ ]:
# region -> LaTeX timing/scaling table -> paper generatedDir (guarded by paper.emit)
if runConfig["plot"]:
    paperT = runConfig.get("paper", {})

    # headline speedup: best batch acceleration over CPU serial at the largest shared N
    batchRows = df[df["mode"] == "batch"]
    bestSpeedup = float(np.nanmax(batchRows["speedup_vs_serial"])) if len(batchRows) else float("nan")
    minSubjBatchCpu = df[(df["device"] == "cpu") & (df["mode"] == "batch")]["s_per_subject"]
    minSubjBatchCpu = float(np.nanmin(minSubjBatchCpu)) if len(minSubjBatchCpu) else float("nan")

    rowsTex = {}
    for i, r in df.iterrows():
        key = f"{r['device']}-{r['mode']}-{int(r['N'])}"
        rowsTex[key] = [
            r["device"], r["mode"], f"{int(r['N'])}",
            f"{r['s_per_subject']:.3g}",
            f"{r['total_wall_s']:.3g}",
            ("--" if not np.isfinite(r["speedup_vs_serial"]) else f"{r['speedup_vs_serial']:.1f}"),
        ]

    timingTable = utils.generate_latex_table_new(
        rowsTex,
        ["Case", "Device", "Mode", "$N$", "s/subject", "Total wall (s)", "Speedup $\\times$"],
        "", "", "timingComparison",
        f"Wall-clock cost and scaling of one {int(sp['runTime'])}~s forward solve of the cardio "
        f"model (the atomic unit of an EFC calibration; a full staged calibration is $K$ such "
        f"solves) run serially, as a batched CPU \\texttt{{vmap}}, and as a batched GPU "
        f"\\texttt{{vmap}}, over a cohort-size sweep $N$. Batching amortizes per-solve dispatch: "
        f"the per-subject cost falls to $\\sim${minSubjBatchCpu:.2g}~s on CPU, a "
        f"$\\sim${bestSpeedup:.0f}$\\times$ speedup over the serial loop. The GPU leg is "
        f"launch-latency bound (a long sequential scan over a small state vector) and does not "
        f"beat CPU for this workload. Speedup is relative to the CPU serial total at the same $N$.")

    if paperT.get("emit", False):
        genDir = paperT.get("generatedDir", "EFC_Paper/revision/generated")
        os.makedirs(genDir, exist_ok=True)
        outTable = os.path.join(genDir, paperT.get("tableName", "timingComparison.tex"))
        with open(outTable, "w") as fh:
            fh.write(timingTable)
        print(f"LaTeX table -> {outTable}\n")
    print(timingTable)
# endregion


## Scaling figure — total wall &amp; s/subject vs $N$

In [ ]:
# region -> scaling figure: total wall (log-log) + s/subject vs N, one line per device+mode
if runConfig["plot"]:
    paperF = runConfig.get("paper", {})
    groups = list(df.groupby(["device", "mode"]))
    colors = [plt.get_cmap("tab10")(k) for k in range(len(groups))]

    fig, (axW, axS) = plt.subplots(1, 2, figsize=(13, 5))
    for (key, g), col in zip(groups, colors):
        g = g.sort_values("N")
        lbl = f"{key[0]}-{key[1]}"
        axW.plot(g["N"], g["total_wall_s"], "o-", color=col, label=lbl)
        axS.plot(g["N"], g["s_per_subject"], "o-", color=col, label=lbl)

    axW.set_xscale("log"); axW.set_yscale("log")
    axW.set_xlabel("cohort size $N$ (subjects)"); axW.set_ylabel("total wall-clock (s)")
    axW.set_title("Total cost vs cohort size"); axW.legend(fontsize=8); axW.grid(True, which="both", alpha=0.3)

    axS.set_xscale("log"); axS.set_yscale("log")
    axS.set_xlabel("cohort size $N$ (subjects)"); axS.set_ylabel("wall-clock per subject (s)")
    axS.set_title("Amortized per-subject cost"); axS.legend(fontsize=8); axS.grid(True, which="both", alpha=0.3)
    plt.tight_layout()

    if paperF.get("emit", False):
        imgDir = paperF.get("imagesDir", "EFC_Paper/revision/Images")
        os.makedirs(imgDir, exist_ok=True)
        outFig = os.path.join(imgDir, paperF.get("figName", "timingComparison.png"))
        plt.savefig(outFig, dpi=200, bbox_inches="tight")
        print(f"figure -> {outFig}")
    plt.show()
# endregion
